In [2]:
import pandas as pd
from pathlib import Path

# Пути к папкам
root = Path('..')  # один уровень вверх от notebooks/
raw = root / 'data' / 'raw'
out = root / 'data' / 'processed'
out.mkdir(parents=True, exist_ok=True)

# ---------- 1) Загружаем воздух ----------
air = pd.read_csv(raw / 'Almaty Air Pollution 2020-2024.csv')
air['date'] = pd.to_datetime(air['date'])
air = air.sort_values('date').set_index('date')

# Если в файле есть колонка "city" — оставляем только Алматы
if 'city' in air.columns:
    mask = air['city'].astype(str).str.contains('Алмат', case=False) | air['city'].astype(str).str.contains('Almat', case=False)
    air = air[mask]

# Переименуем столбцы под удобные имена
rename_map = {'PM2.5':'pm25','PM10':'pm10','NO2':'no2','SO2':'so2','CO':'co','AQI':'aqi'}
air = air.rename(columns=rename_map)

# Делаем дневной ресемплинг
num_cols = [c for c in ['aqi','pm25','pm10','no2','so2','co'] if c in air.columns]
air_daily = air[num_cols].resample('D').mean()

# ---------- 2) Сохраняем готовый файл ----------
out_file = out / 'almaty_daily.csv'
air_daily.to_csv(out_file)
print(" Файл сохранён:", out_file.resolve())
air_daily.head()

 Файл сохранён: /Users/rasulvalitov/Desktop/ml diplom/data/processed/almaty_daily.csv


""
date
2020-02-10
2020-02-11
2020-02-12
2020-02-13
2020-02-14


In [1]:
import pandas as pd
from pathlib import Path

root = Path('..')
raw = root / 'data' / 'raw'
out = root / 'data' / 'processed'

# Загружаем уже сохранённый дневной воздух
df = pd.read_csv(out / 'almaty_daily.csv', parse_dates=['date']).set_index('date').sort_index()

def load_yearly_xls(path: Path, value_col=None, city_col=None, year_col=None):
    """
    Читает .xls/.xlsx с годовыми показателями по регионам,
    оставляет строки по г. Алматы, апсемплирует до дневной частоты (ffill).
    """
    ext = path.suffix.lower()
    engine = 'xlrd' if ext == '.xls' else 'openpyxl'

    # читаем файл
    try:
        tmp = pd.read_excel(path, engine=engine)
    except Exception as e:
        # запасной путь: вдруг расширение не совпадает с реальным форматом
        alt_engine = 'openpyxl' if engine == 'xlrd' else 'xlrd'
        try:
            tmp = pd.read_excel(path, engine=alt_engine)
        except Exception as e2:
            raise RuntimeError(f'Не удалось прочитать {path.name}: {e} / {e2}')

    # угадываем колонки
    cols_l = {c.lower(): c for c in tmp.columns}
    if city_col is None:
        for guess in ['регион','город','область','region','city']:
            if guess in cols_l: city_col = cols_l[guess]; break
    if year_col is None:
        for guess in ['год','year']:
            if guess in cols_l: year_col = cols_l[guess]; break
    if value_col is None:
        numeric = [c for c in tmp.columns if c != year_col and pd.api.types.is_numeric_dtype(tmp[c])]
        if not numeric:
            raise ValueError(f'В {path.name} не нашли числовых колонок')
        value_col = numeric[0]

    # фильтруем г. Алматы
    if city_col and city_col in tmp.columns:
        m = tmp[city_col].astype(str).str.contains('Алмат', case=False) | \
            tmp[city_col].astype(str).str.contains('Almat', case=False)
        tmp = tmp[m]

    tmp = tmp[[year_col, value_col]].dropna()
    tmp[year_col] = pd.to_datetime(tmp[year_col].astype(int), format='%Y')
    tmp = tmp.set_index(year_col).sort_index()

    # апсемплинг до дней
    daily = tmp.resample('D').ffill()
    return daily.rename(columns={value_col: path.stem})

# перечисли свои xls/xlsx (как они лежат у тебя в data/raw)
xls_files = [
    'Выбросы_твердых_загрязняющих_веществ.xls',
    'Выбросы_жидких_и_газообразобразных_загрязняющих_веществ.xls',  # оставил точное имя файла
    'Выбросы_загрязн_атмосферу__веществотходящих_от_стац-х_источн(на_душу_населения).xls',
    'Количество_стационарных_источников_загрязнения.xls',
]

added = []
for fname in xls_files:
    p = raw / fname
    if p.exists():
        try:
            feat = load_yearly_xls(p)
            df = df.join(feat, how='left')
            added.append(fname)
        except Exception as e:
            print(f'⚠️ {fname}: {e}')
    else:
        print(f'⚠️ Нет файла: {fname}')

print('Добавили признаков из файлов:', added)

final_path = out / 'almaty_daily_merged.csv'
df.to_csv(final_path)
print('✅ Сохранён итоговый датасет:', final_path.resolve())
df.head(), df.columns.tolist()


⚠️ Выбросы_твердых_загрязняющих_веществ.xls: В Выбросы_твердых_загрязняющих_веществ.xls не нашли числовых колонок
⚠️ Выбросы_жидких_и_газообразобразных_загрязняющих_веществ.xls: В Выбросы_жидких_и_газообразобразных_загрязняющих_веществ.xls не нашли числовых колонок
⚠️ Выбросы_загрязн_атмосферу__веществотходящих_от_стац-х_источн(на_душу_населения).xls: В Выбросы_загрязн_атмосферу__веществотходящих_от_стац-х_источн(на_душу_населения).xls не нашли числовых колонок
⚠️ Количество_стационарных_источников_загрязнения.xls: В Количество_стационарных_источников_загрязнения.xls не нашли числовых колонок
Добавили признаков из файлов: []
✅ Сохранён итоговый датасет: /Users/rasulvalitov/Desktop/ml diplom/data/processed/almaty_daily_merged.csv


(Empty DataFrame
 Columns: []
 Index: [2020-02-10 00:00:00, 2020-02-11 00:00:00, 2020-02-12 00:00:00, 2020-02-13 00:00:00, 2020-02-14 00:00:00],
 [])

In [ ]:
final_path = out / 'almaty_daily_merged.csv'
df.to_csv(final_path)
print('✅ Сохранён итоговый датасет:', final_path.resolve())

In [2]:
import pandas as pd, numpy as np, re
from pathlib import Path

root = Path('..')
raw = root/'data'/'raw'
out = root/'data'/'processed'
out.mkdir(parents=True, exist_ok=True)

# ----- Карта файлов -> человекочитаемая категория и единицы -----
FILE_META = {
    'Количество_стационарных_источников_загрязнения.xls': {
        'category': 'Количество источников выбросов загрязняющих веществ',
        'unit': 'шт.'
    },
    'Выбросы_твердых_загрязняющих_веществ.xls': {
        'category': 'Выбросы твёрдых загрязняющих веществ',
        'unit': 'тыс. тонн'
    },
    'Выбросы_загрязн_атмосферу__веществотходящих_от_стац-х_источн(на_душу_населения).xls': {
        'category': 'Выбросы загрязняющих веществ от стац. источников (на душу населения)',
        'unit': 'кг/чел'
    },
    # ⚠️ здесь учтён файл с двойным "об" в названии
    'Выбросы_жидких_и_газообразобразных_загрязняющих_веществ.xls': {
        'category': 'Выбросы жидких и газообразных загрязняющих веществ',
        'unit': 'тыс. тонн'
    },
}

YEAR_MIN, YEAR_MAX = 2005, 2024
YEAR_RE = re.compile(r'^(20\d{2})$')

def _clean_num(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float, np.number)): return float(x)
    s = str(x).strip().replace('\xa0','').replace(' ','').replace(',', '.')
    if s in {'', '-', '–', '—'}: return np.nan
    try:
        return float(s)
    except:
        s2 = re.sub(r'[^\d\.]', '', s)
        try: return float(s2)
        except: return np.nan

def extract_almaty_from_sheet(df_sheet: pd.DataFrame):
    df = df_sheet.copy()
    df = df.rename(columns=lambda c: str(c).strip())

    # 1) строка "г. Алматы" (не область)
    row_idx = None
    for i in range(len(df)):
        v = df.iloc[i, 0]
        if isinstance(v, str):
            t = v.lower().replace(' ', '')
            if ('алмат' in t) and ('обл' not in t):
                row_idx = i; break
    if row_idx is None:
        return {}

    # 2) столбцы с годами (из имён и верхних строк)
    col_years = {}
    for j, colname in enumerate(df.columns):
        m = YEAR_RE.match(str(colname).strip())
        if m:
            y = int(m.group(1))
            if YEAR_MIN <= y <= YEAR_MAX:
                col_years[y] = df.columns[j]

    if len(col_years) < 5:
        header_rows_to_check = min(5, len(df))
        for j in range(df.shape[1]):
            for i in range(header_rows_to_check):
                val = df.iat[i, j]
                m = YEAR_RE.match(str(val).strip())
                if m:
                    y = int(m.group(1))
                    if YEAR_MIN <= y <= YEAR_MAX and y not in col_years:
                        col_years[y] = df.columns[j]

    # 3) значения по строке Алматы
    series = {}
    for y, col in sorted(col_years.items()):
        series[y] = _clean_num(df.at[row_idx, col])
    return series

def read_any_excel(path: Path):
    engine = 'xlrd' if path.suffix.lower()=='.xls' else 'openpyxl'
    try:
        xls = pd.ExcelFile(path, engine=engine)
    except Exception:
        alt = 'openpyxl' if engine=='xlrd' else 'xlrd'
        xls = pd.ExcelFile(path, engine=alt)

    agg = {}
    for sheet in xls.sheet_names:
        df_sheet = pd.read_excel(xls, sheet_name=sheet, header=None)
        got = extract_almaty_from_sheet(df_sheet)
        for y, v in got.items():
            if y not in agg or (pd.isna(agg[y]) and not pd.isna(v)):
                agg[y] = v
    return agg

# ----- обходим все файлы и складываем tidy-таблицу -----
rows = []
for fname, meta in FILE_META.items():
    p = raw/fname
    if not p.exists():
        print(f'⚠️ Нет файла: {fname}')
        continue
    series = read_any_excel(p)
    if not series:
        print(f'⚠️ Не удалось извлечь данные из: {fname}')
        continue
    for y, v in series.items():
        if pd.isna(v): 
            continue
        rows.append({
            'category': meta['category'],
            'unit': meta['unit'],
            'year': int(y),
            'value': float(v)
        })

yearly = pd.DataFrame(rows).sort_values(['category','year']).reset_index(drop=True)
print('Найдено строк:', len(yearly))
display(yearly.head(10))

yearly_path = out/'almaty_yearly_from_excels.csv'
yearly.to_csv(yearly_path, index=False)
print('✅ Сохранено:', yearly_path.resolve())
# склейка в единый дневной датасет
air_daily = pd.read_csv(out/'almaty_daily.csv', parse_dates=['date']).set_index('date').sort_index()
yearly = pd.read_csv(out/'almaty_yearly_from_excels.csv')
yearly['cat_unit'] = yearly['category'] + ' [' + yearly['unit'] + ']'

wide = yearly.pivot(index='year', columns='cat_unit', values='value')
wide.index = pd.to_datetime(wide.index, format='%Y')
wide_daily = wide.resample('D').ffill()

merged = air_daily.join(wide_daily, how='left')
merged['month'] = merged.index.month
merged['dow']   = merged.index.dayofweek

merged_path = out/'almaty_daily_plus_yearly.csv'
merged.to_csv(merged_path)
print('✅ Готов финальный датасет:', merged_path.resolve())


Найдено строк: 72


,category,unit,year,value
0,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2005,44.8
1,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2006,48.6
2,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2007,40.5
3,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2008,45.7
4,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2009,48.0
5,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2010,51.4
6,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2011,53.3
7,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2012,48.0
8,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2013,52.8
9,Выбросы жидких и газообразных загрязняющих вещ...,тыс. тонн,2014,39.7


✅ Сохранено: /Users/rasulvalitov/Desktop/ml diplom/data/processed/almaty_yearly_from_excels.csv


In [3]:
# склейка в единый дневной датасет
air_daily = pd.read_csv(out/'almaty_daily.csv', parse_dates=['date']).set_index('date').sort_index()
yearly = pd.read_csv(out/'almaty_yearly_from_excels.csv')
yearly['cat_unit'] = yearly['category'] + ' [' + yearly['unit'] + ']'

wide = yearly.pivot(index='year', columns='cat_unit', values='value')
wide.index = pd.to_datetime(wide.index, format='%Y')
wide_daily = wide.resample('D').ffill()

merged = air_daily.join(wide_daily, how='left')
merged['month'] = merged.index.month
merged['dow']   = merged.index.dayofweek

merged_path = out/'almaty_daily_plus_yearly.csv'
merged.to_csv(merged_path)
print('✅ Готов финальный датасет:', merged_path.resolve())


✅ Готов финальный датасет: /Users/rasulvalitov/Desktop/ml diplom/data/processed/almaty_daily_plus_yearly.csv


In [6]:
import pandas as pd, numpy as np, re
from pathlib import Path

# ---------- Пути ----------
project = Path('..') if (Path('..')/'data'/'raw').exists() else Path('.')
raw = project/'data'/'raw'
out = project/'data'/'processed'
out.mkdir(parents=True, exist_ok=True)

# Автопоиск файла по имени
cands = sorted([p for p in raw.glob('*.xls*') if re.search(r'налич.*автотранспорт', p.name, re.I)])
if not cands:
    raise FileNotFoundError(f'В {raw} нет файла про "Наличие автотранспортных средств". Найдено: {[p.name for p in raw.glob("*")]}')
file_path = cands[-1]
print('📄 Использую файл:', file_path.name)

# ---------- Константы ----------
YEAR_MIN, YEAR_MAX = 1992, 2035
KEEP_FROM, KEEP_TO = 2003, 2024
ALMATY_RGX = re.compile(r'(^|\s)(г\.?\s*)?алмат(?!инск)', re.I)  # г. Алматы/Алматы, НЕ алматинск*
TITLES = {
    r'Всего\s+автотранспортных\s+средств' : ('Всего автотранспортных средств', 'тыс. ед.'),
    r'Легковые\s+автомобили'               : ('Легковые автомобили',             'тыс. ед.'),
    r'Грузовые\s+автомобили'               : ('Грузовые автомобили',             'тыс. ед.'),
    r'Автобусы'                            : ('Автобусы',                        'тыс. ед.'),
}
TITLE_RGX = re.compile('|'.join(TITLES.keys()), re.I)
DEBUG = True

# ---------- Вспомогательные ----------
def _is_year(v):
    if pd.isna(v): return None
    if isinstance(v,(int,np.integer)):
        y=int(v);  return y if YEAR_MIN<=y<=YEAR_MAX else None
    if isinstance(v,(float,np.floating)):
        y=int(round(float(v)))
        return y if abs(float(v)-y)<1e-6 and YEAR_MIN<=y<=YEAR_MAX else None
    s=str(v).strip()
    m=re.match(r'^\s*(20\d{2})(?:[.,]0+)?\s*$', s)
    return int(m.group(1)) if m and YEAR_MIN<=int(m.group(1))<=YEAR_MAX else None

def _row_years(vals):
    out={}
    for j,val in enumerate(vals):
        y=_is_year(val)
        if y is not None: out[j]=y
    return out

def _clean_num(x):
    if pd.isna(x): return np.nan
    if isinstance(x,(int,float,np.number)): return float(x)
    s=str(x).replace('\xa0','').replace(' ','').replace(',','.')
    if s in {'','-','–','—'}: return np.nan
    try: return float(s)
    except: 
        s2=re.sub(r'[^\d\.-]','',s)
        return float(s2) if s2 not in {'','-','.'} else np.nan

def _find_title_rows(df):
    rows=[]
    for i in range(len(df)):
        row_txt=' | '.join([str(x) for x in df.iloc[i].tolist()])
        if TITLE_RGX.search(row_txt):
            for rgx in TITLES:
                if re.search(rgx,row_txt,re.I):
                    rows.append((i,rgx))
                    break
    return sorted(rows,key=lambda t:t[0])

def _years_cols_local(df, seg_start, seg_end, scan_rows=120, min_years=3):
    year2col={}
    for i in range(seg_start, min(seg_start+scan_rows, seg_end)):
        yr=_row_years(df.iloc[i].tolist())
        for j in sorted(yr): 
            y=yr[j]
            if KEEP_FROM<=y<=KEEP_TO and y not in year2col:
                year2col[y]=j
    return dict(sorted(year2col.items())) if len(set(year2col))>=min_years else {}

def _years_cols_global(df, top_rows=60, min_years=10):
    """Глобальный поиск строки годов в первых top_rows строках всей таблицы."""
    best=None
    for i in range(min(top_rows, len(df))):
        yr=_row_years(df.iloc[i].tolist())
        yrs=list(sorted(yr.values()))
        uniq=len(set([y for y in yrs if KEEP_FROM<=y<=KEEP_TO]))
        if uniq>=min_years:
            # нормируем к диапазону KEEP_FROM..KEEP_TO
            year2col={y:col for col,y in {col:y for col,y in yr.items()}.items() if KEEP_FROM<=y<=KEEP_TO}
            best=dict(sorted(year2col.items()))
            break
    return best or {}

def _find_almaty_row(df, seg_start, seg_end, expand=10):
    # сначала в сегменте
    for r in range(seg_start, seg_end):
        row_txt=' '.join([str(x) for x in df.iloc[r].tolist()])
        if ALMATY_RGX.search(row_txt) and 'алматинск' not in row_txt.lower():
            return r
    # немного расширим вниз (бывает, что «Алматы» начинается сразу после блока)
    for r in range(seg_end, min(seg_end+expand, len(df))):
        row_txt=' '.join([str(x) for x in df.iloc[r].tolist()])
        if ALMATY_RGX.search(row_txt) and 'алматинск' not in row_txt.lower():
            return r
    return None

# ---------- Парсинг ----------
engine = 'openpyxl' if file_path.suffix.lower()=='.xlsx' else None
xls = pd.ExcelFile(file_path, engine=engine)

all_rows=[]
for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet, header=None)
    titles=_find_title_rows(df)
    if not titles: 
        continue
    # сегменты
    segments=[]
    for k,(r,rgx) in enumerate(titles):
        start=r
        end=titles[k+1][0] if k+1<len(titles) else len(df)
        segments.append((start,end,rgx))
    # глобальные годы (на случай «общей шапки»)
    global_years=_years_cols_global(df, top_rows=80, min_years=8)
    if DEBUG:
        print(f'[{sheet}] глобальные годы найдены: {list(global_years.keys())[:6]}{"..." if len(global_years)>6 else ""}' if global_years else f'[{sheet}] глобальные годы НЕ найдены')

    for seg_start, seg_end, rgx in segments:
        cat,unit = TITLES[rgx]
        if DEBUG: print(f'[{sheet}] ▶️ Сегмент "{cat}": {seg_start}-{seg_end}')
        # 1) локально
        year2col = _years_cols_local(df, seg_start, seg_end, scan_rows=160, min_years=3)
        # 2) если пусто — используем глобальные
        if not year2col and global_years:
            year2col = global_years
            if DEBUG: print(f'[{sheet}]   (fallback) берём глобальную строку годов')
        if not year2col:
            if DEBUG: print(f'[{sheet}]   ⚠️ годы не найдены')
            continue
        # 3) ищем строку Алматы
        r_alma = _find_almaty_row(df, seg_start, seg_end, expand=8)
        if r_alma is None:
            if DEBUG: print(f'[{sheet}]   ⚠️ строка "г. Алматы" не найдена')
            continue
        # 4) собираем значения
        for y,col in sorted(year2col.items()):
            if not (KEEP_FROM<=y<=KEEP_TO): 
                continue
            val = _clean_num(df.iat[r_alma, col])
            if not np.isnan(val):
                all_rows.append({'year':int(y),'category':cat,'unit':unit,'value':float(val)})

if not all_rows:
    raise RuntimeError('Не удалось извлечь ни одной строки — похоже, годы в файле совсем не распознаются как числа/строки. '
                       'Пришли, пожалуйста, один скрин: заголовок блока + сами годы (ряд с “2003 … 2024”).')

# ---------- Сохранение ----------
res = (pd.DataFrame(all_rows)
         .drop_duplicates()
         .sort_values(['category','year'])
         .reset_index(drop=True))

save_path = out/'almaty_transport_yearly.csv'
res.to_csv(save_path, index=False, encoding='utf-8-sig')
print('✅ Сохранено:', save_path.resolve())
display(res.groupby('category')['year'].agg(['min','max','count']))
display(res.head(12))


📄 Использую файл: Наличие автотранспортных средств.xlsx
[Наличие автотранспорта] глобальные годы найдены: [2003, 2004, 2005, 2006, 2007, 2008]...
[Наличие автотранспорта] ▶️ Сегмент "Всего автотранспортных средств": 5-58
[Наличие автотранспорта]   (fallback) берём глобальную строку годов
[Наличие автотранспорта] ▶️ Сегмент "Легковые автомобили": 58-110
[Наличие автотранспорта]   (fallback) берём глобальную строку годов
[Наличие автотранспорта] ▶️ Сегмент "Грузовые автомобили": 110-162
[Наличие автотранспорта]   (fallback) берём глобальную строку годов
[Наличие автотранспорта] ▶️ Сегмент "Автобусы": 162-218
[Наличие автотранспорта]   (fallback) берём глобальную строку годов
✅ Сохранено: /Users/rasulvalitov/Desktop/ml diplom/data/processed/almaty_transport_yearly.csv


,min,max,count
category,,,
Автобусы,2003,2024,22
Всего автотранспортных средств,2003,2024,22
Грузовые автомобили,2003,2024,22
Легковые автомобили,2003,2024,22


,year,category,unit,value
0,2003,Автобусы,тыс. ед.,11.888
1,2004,Автобусы,тыс. ед.,9.562
2,2005,Автобусы,тыс. ед.,8.107
3,2006,Автобусы,тыс. ед.,8.737
4,2007,Автобусы,тыс. ед.,9.576
5,2008,Автобусы,тыс. ед.,9.838
6,2009,Автобусы,тыс. ед.,12.450
7,2010,Автобусы,тыс. ед.,12.153
8,2011,Автобусы,тыс. ед.,11.434
9,2012,Автобусы,тыс. ед.,10.753


In [2]:
import pandas as pd, numpy as np, re
from pathlib import Path

# ===================== ПУТИ =====================
project = Path('..') if (Path('..')/'data'/'raw').exists() else Path('.')
raw = project/'data'/'raw'
out = project/'data'/'processed'
out.mkdir(parents=True, exist_ok=True)

print('📂 RAW:', raw.resolve())
print('💾 PROCESSED:', out.resolve())

# ===================== ПОМОЩНИКИ =====================
def coerce_float(x):
    """Строковое число -> float: пробелы, неразрывные пробелы, запятые, лишние символы."""
    if pd.isna(x): return np.nan
    s = str(x).replace('\xa0','').replace(' ','').replace(',','.')
    try:
        return float(s)
    except:
        s2 = re.sub(r'[^\d\.\-]', '', s)
        return float(s2) if s2 not in {'', '-', '.'} else np.nan

def find_csv_by_topic():
    """
    Автопоиск двух CSV в ../data/raw:
    - одна про 'Численность населения' (population)
    - вторая про 'Естественный прирост' (natural growth)
    Оба — это выгрузки stat.gov.kz с колонками типа: КАТО(по каталогу), NAM, DAT, VAL
    """
    csvs = sorted(raw.glob('*.csv'))
    if not csvs:
        raise FileNotFoundError(f'В {raw} нет CSV. Положи выгрузки stat.gov.kz (CSV, табуляция).')

    pop_path, ng_path = None, None
    for p in csvs:
        try:
            df = pd.read_csv(p, sep='\t', nrows=200)
        except Exception:
            continue
        cols = {c.lower():c for c in df.columns}
        # проверим поле NAM — обычно там есть текстовый "заголовок" показателя
        if 'nam' in cols:
            nam_vals = ' '.join(df[cols['nam']].astype(str).head(50).tolist()).lower()
            if re.search(r'численност|population|населен', nam_vals):
                pop_path = pop_path or p
            if re.search(r'естествен.*прирост|ест.*убыль|natural', nam_vals):
                ng_path = ng_path or p

    # fallback: если названия не распознали по NAM, пытаемся по имени файла
    for p in csvs:
        name = p.name.lower()
        if pop_path is None and re.search(r'pop|численн|населен', name): pop_path = p
        if ng_path  is None and re.search(r'прирост|убыль|growth', name): ng_path  = p

    if pop_path is None or ng_path is None:
        raise FileNotFoundError(
            f'Не удалось однозначно определить CSV. Нашёл: {[c.name for c in csvs]}\n'
            f'Подпиши файлы понятнее или проверь, что в колонке NAM есть ключевые слова.'
        )
    return pop_path, ng_path

def read_stat_csv(path):
    """Чтение CSV stat.gov.kz с разделителем табуляции и чисткой заголовков."""
    df = pd.read_csv(path, sep='\t', engine='python')
    df.columns = [str(c).strip() for c in df.columns]
    return df

def extract_year_series(df_city):
    """
    Извлечь пару (year, value) из типовой выгрузки stat.gov.kz:
    - год в колонке DAT (или в любой строке, где встречается 4 цифры подряд)
    - значение в колонке VAL (последняя числовая колонка как fallback)
    """
    # Год
    year = None
    if 'DAT' in df_city.columns:
        year = pd.to_numeric(df_city['DAT'].astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
    else:
        year = df_city.apply(lambda r: pd.to_numeric(re.search(r'(\d{4})', ' '.join(map(str,r.values)) or "").group(1))
                             if re.search(r'(\d{4})', ' '.join(map(str,r.values)) or "") else np.nan, axis=1)
    df_city = df_city.assign(year=year)

    # Значение
    val_col = 'VAL' if 'VAL' in df_city.columns else None
    if val_col is None:
        # берём последний столбец, где есть числовые значения
        cand = None
        for c in df_city.columns[::-1]:
            vals = pd.to_numeric(df_city[c].astype(str).str.replace('\xa0','').str.replace(' ','').str.replace(',','.'), errors='coerce')
            if vals.notna().sum() > 0:
                cand = c; break
        val_col = cand
    if val_col is None:
        raise ValueError('Не найден столбец со значением (VAL).')

    values = df_city[val_col].apply(coerce_float)
    return df_city[['year']].assign(value=values)

# ===================== 1) CSV: Population & Natural growth (Г.АЛМАТЫ) =====================
pop_csv, ng_csv = find_csv_by_topic()
print('👀 Найдены CSV:', pop_csv.name, '|', ng_csv.name)

pop_raw = read_stat_csv(pop_csv)
ng_raw  = read_stat_csv(ng_csv)

# фильтр по КАТО: Г.АЛМАТЫ
if 'КАТО(по каталогу)' not in pop_raw.columns or 'КАТО(по каталогу)' not in ng_raw.columns:
    raise KeyError('В CSV нет колонки "КАТО(по каталогу)". Проверь выгрузку stat.gov.kz.')

pop_city = pop_raw[pop_raw['КАТО(по каталогу)'].astype(str).str.contains('Г.АЛМАТЫ', na=False)].copy()
ng_city  = ng_raw [ng_raw ['КАТО(по каталогу)'].astype(str).str.contains('Г.АЛМАТЫ', na=False)].copy()

# проверим, что это реально нужные показатели (опционально)
# print(pop_city['NAM'].head(3).to_list())
# print(ng_city['NAM'].head(3).to_list())

# извлекаем год-значение
pop_series = extract_year_series(pop_city).dropna().drop_duplicates().sort_values('year')
ng_series  = extract_year_series(ng_city ).dropna().drop_duplicates().sort_values('year')

pop_series = pop_series.rename(columns={'value':'population'})
ng_series  = ng_series.rename(columns={'value':'natural_growth_year'})

# ===================== 2) Excel: Естественный прирост/убыль по месяцам (АГРЕГИРУЕМ В ГОД) =====================
# найдём xls/xlsx
monthly_xls = None
for p in sorted(raw.glob('*.xls*')):
    if re.search(r'естествен.*прирост|убыль', p.name, re.I):
        monthly_xls = p
        break
if monthly_xls is None:
    # если не нашли по названию, возьмём самый крупный xls/xlsx
    cand = sorted(raw.glob('*.xls*'), key=lambda x: x.stat().st_size, reverse=True)
    monthly_xls = cand[0] if cand else None

monthly_df = pd.DataFrame(columns=['year','month','natural_growth_month'])
if monthly_xls is not None:
    print('📄 Excel (monthly):', monthly_xls.name)
    xls = pd.ExcelFile(monthly_xls)
    dfm = pd.read_excel(xls, xls.sheet_names[0], header=None)

    # строка с "г. Алматы" (не область)
    alma_row = None
    for i in range(len(dfm)):
        row_txt = ' '.join([str(x) for x in dfm.iloc[i].tolist()]).lower()
        if ('алмат' in row_txt) and ('обл' not in row_txt):
            alma_row = i
            break

    months_ru = {"январь":1,"февраль":2,"март":3,"апрель":4,"май":5,"июнь":6,
                 "июль":7,"август":8,"сентябрь":9,"октябрь":10,"ноябрь":11,"декабрь":12}

    def find_month_for_col(c, search_rows=80):
        if alma_row is None: return None, None
        # ищем подпись месяца "вверх" по колонке
        for r in range(max(0, alma_row-search_rows), alma_row+1):
            s = str(dfm.iat[r, c]).strip().lower() if c < dfm.shape[1] else ""
            if s in months_ru: return months_ru[s], r
        # и в верхних строках
        for r in range(0, min(100, len(dfm))):
            s = str(dfm.iat[r, c]).strip().lower() if c < dfm.shape[1] else ""
            if s in months_ru: return months_ru[s], r
        return None, None

    def find_year_above(col, row_from, max_up=40):
        for r in range(max(0, row_from-max_up), row_from+1):
            txt = str(dfm.iat[r, col]) if col < dfm.shape[1] else ""
            m = re.search(r'(20\d{2})', txt)
            if m:
                y = int(m.group(1))
                if 1990 <= y <= 2035: return y
        for r in range(0, min(15, len(dfm))):
            txt = str(dfm.iat[r, col]) if col < dfm.shape[1] else ""
            m = re.search(r'(20\d{2})', txt)
            if m:
                y = int(m.group(1))
                if 1990 <= y <= 2035: return y
        return None

    rows = []
    if alma_row is not None:
        for c in range(dfm.shape[1]):
            v = coerce_float(dfm.iat[alma_row, c])
            if pd.isna(v): 
                continue
            m, m_row = find_month_for_col(c, search_rows=100)
            if m is None: 
                continue
            y = find_year_above(c, m_row if m_row is not None else alma_row, max_up=40)
            if y is None:
                # иногда год в соседней колонке
                for dc in (-1,1,-2,2):
                    cc = c+dc
                    if 0 <= cc < dfm.shape[1]:
                        y = find_year_above(cc, m_row if m_row is not None else alma_row, max_up=40)
                        if y: break
            if y is None: 
                continue
            rows.append({'year':int(y),'month':int(m),'natural_growth_month':float(v)})

    monthly_df = (pd.DataFrame(rows)
                    .drop_duplicates()
                    .sort_values(['year','month']))

# агрегируем в год
ng_from_months = (monthly_df.groupby('year', as_index=False)
                    .agg(natural_growth_year_from_months=('natural_growth_month','sum'),
                         months_available=('month','nunique')))

# ===================== 3) ЕДИНЫЙ CSV ПО ГОДАМ =====================
merged = (pop_series
          .merge(ng_series, on='year', how='outer')
          .merge(ng_from_months, on='year', how='outer')
          .sort_values('year')
          .reset_index(drop=True))

# удаляем полностью пустые годы (если такое вдруг появится)
value_cols = ['population','natural_growth_year','natural_growth_year_from_months']
merged = merged.dropna(subset=value_cols, how='all')

# сохраняем
out_path = out/'almaty_population_all.csv'
merged.to_csv(out_path, index=False, encoding='utf-8-sig')

print('✅ Сохранено:', out_path.resolve())
display(merged.head(15))
display(merged.tail(10))


📂 RAW: /Users/rasulvalitov/Desktop/ml diplom/data/raw
💾 PROCESSED: /Users/rasulvalitov/Desktop/ml diplom/data/processed
👀 Найдены CSV: data-2.csv | data-3.csv
📄 Excel (monthly): Естественный прирост (убыль) населения.xlsx
✅ Сохранено: /Users/rasulvalitov/Desktop/ml diplom/data/processed/almaty_population_all.csv


,year,population,natural_growth_year,natural_growth_year_from_months,months_available
0,2000,1130439.0,2717.0,NaN,NaN
1,2001,1128759.0,1867.0,NaN,NaN
2,2002,1132424.0,4195.0,NaN,NaN
3,2003,1149641.0,6768.0,NaN,NaN
4,2004,1175208.0,10591.0,NaN,NaN
5,2005,1209485.0,12202.0,NaN,NaN
6,2006,1247896.0,15920.0,NaN,NaN
7,2007,1287246.0,17577.0,NaN,NaN
8,2008,1324739.0,21815.0,NaN,NaN
9,2009,1361877.0,14285.0,7665.0,4.0


,year,population,natural_growth_year,natural_growth_year_from_months,months_available
16,2016,1702766.0,20910.0,NaN,NaN
17,2017,1751308.0,20488.0,NaN,NaN
18,2018,1801993.0,21581.0,NaN,NaN
19,2019,1854656.0,22068.0,3811.0,2.0
20,2020,1916822.0,21086.0,NaN,NaN
21,2021,1977258.0,19702.0,NaN,NaN
22,2022,2101485.0,23784.0,NaN,NaN
23,2023,2161902.0,23565.0,NaN,NaN
24,2024,2228677.0,22196.0,2000.0,1.0
25,2025,2292055.0,NaN,7856.0,5.0


In [11]:
import pandas as pd, numpy as np, re
from pathlib import Path

# --- пути
root = Path('..')
raw = root/'data'/'raw'
out = root/'data'/'processed'
out.mkdir(parents=True, exist_ok=True)

# --- общие регэкспы
YEAR_RE = re.compile(r'(19|20)\d{2}')
MONTH_SHEET_RE = re.compile(r'^(0?[1-9]|1[0-2])$')  # имена листов "01".."12"
ALMATY_RE = re.compile(r'(^|[^а-я])г?\.?\s*алматы\b', flags=re.I)
ALMATY_REGION_RE = re.compile(r'алматинск', flags=re.I)

def clean_num(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    if s in {'', '-', '–', '—', '…'}: return np.nan
    s = s.replace('\xa0','').replace('\u2009','')  # тонкие/неразрывные пробелы
    s = s.replace('%','')                           # убираем проценты из числа
    s = s.replace(',', '.')                         # запятая -> точка
    s = re.sub(r'[^0-9\.\-eE]', '', s)              # оставим только число
    try: return float(s)
    except: return np.nan

def guess_unit_from_text(txt: str) -> str:
    t = (txt or '').lower()
    if 'процент' in t or '%' in t: return '%'
    if 'млрд' in t: return 'млрд тг'
    if 'млн' in t: return 'млн тг'
    if 'тыс' in t and 'тг' in t: return 'тыс тг'
    if 'тенге' in t or 'тг' in t: return 'тг'
    if 'квт' in t or 'кВт' in txt: return 'кВт·ч'
    return ''

def year_from_text(*texts):
    for t in texts:
        if not t: continue
        m_all = YEAR_RE.findall(str(t))
        if m_all:
            # берём последнюю 4-значную подстроку (часто в названии есть и 2024, и 2025)
            m = re.findall(r'(?:19|20)\d{2}', str(t))
            if m: return int(m[-1])
    return None

def looks_like_almaty(s: str) -> bool:
    if not isinstance(s, str): return False
    if ALMATY_REGION_RE.search(s): return False  # «Алматинская (обл.)» — мимо
    return bool(ALMATY_RE.search(s))

def find_almaty_cell(df: pd.DataFrame, max_rows=500, max_cols=200):
    R = min(df.shape[0], max_rows)
    C = min(df.shape[1], max_cols)
    best = None
    for i in range(R):
        for j in range(C):
            v = df.iat[i, j]
            if isinstance(v, str) and looks_like_almaty(v):
                # берём с минимальным j (обычно это колонка с названиями регионов)
                if best is None or j < best[1] or (j == best[1] and i < best[0]):
                    best = (i, j)
    return best  # (row_i, col_j) или None

def header_string_for_col(df: pd.DataFrame, col: int, header_rows=15):
    top = df.iloc[:min(header_rows, df.shape[0]), col].astype(str).tolist()
    parts = [t.strip() for t in top if t and t.strip() not in {'', 'nan'}]
    # склеим уникальные подписи, уберём явные года, повторяющиеся «Unnamed»
    parts = [p for p in parts if not YEAR_RE.fullmatch(p)]
    # уберём мусор «Unnamed: X»
    parts = [re.sub(r'^Unnamed.*', '', p, flags=re.I).strip() for p in parts]
    parts = [p for p in parts if p]
    # сократим длину
    txt = ' / '.join(dict.fromkeys(parts))  # сохраняем порядок, убираем дубли
    return txt[:200]

def find_years_in_header(df: pd.DataFrame, header_rows=15):
    """
    Ищем годы в шапке (по колонкам). Возвращаем словарь col_index -> year(int).
    Если шапка без лет (срез за один год), вернём пусто.
    """
    years = {}
    top = df.iloc[:min(header_rows, df.shape[0]), :]
    for j in range(df.shape[1]):
        vals = top.iloc[:, j].astype(str).str.extract(r'((?:19|20)\d{2})', expand=False).dropna()
        if not vals.empty:
            # возьмём последний годовой токен для колонки
            years[j] = int(vals.iloc[-1])
    # если нашли менее чем в 2 колонках — скорее всего это подписи уровня/примечаний → считаем, что "лет в шапке нет"
    if len(years) < 2:
        return {}
    return years

def extract_sheet(df_raw: pd.DataFrame, source_file: str, sheet: str, category_hint: str):
    """
    Универсальный извлекатель:
    - ищет ячейку «г. Алматы» где угодно;
    - если в шапке есть годы — раскладывает по годам;
    - если в шапке годов нет — подставляет год из имени файла/листа (один на лист).
    """
    df = df_raw.copy()
    # никакого header — ищем сами
    df.columns = [str(c) for c in range(df.shape[1])]  # безопасные имена
    pos = find_almaty_cell(df)
    if not pos:
        return [], 'нет строки с «г. Алматы»'
    i_row, j_name = pos

    # годы в шапке?
    col_year = find_years_in_header(df)
    header_txt_all = ' | '.join(df.iloc[:15, :].astype(str).fillna('').values.ravel().tolist())
    unit_from_header = guess_unit_from_text(header_txt_all)

    recs = []

    if col_year:  # классика: годы по колонкам
        for j, yy in col_year.items():
            if j <= j_name:  # до названия региона — не данные
                continue
            val = clean_num(df.iat[i_row, j])
            if pd.isna(val):
                continue
            col_label = header_string_for_col(df, j)
            unit = guess_unit_from_text(col_label) or unit_from_header
            recs.append({
                'kind': category_hint,
                'year': int(yy),
                'variable': col_label or 'показатель',
                'value': float(val),
                'unit': unit,
                'source_file': source_file,
                'sheet': sheet,
                'layout': 'years_in_columns'
            })
        reason = ''
    else:
        # срез на один год — достанем год из имени файла/листа
        yy = year_from_text(source_file, sheet)
        if yy is None:
            return [], 'нет годов в шапке и в названии файла/листа'
        for j in range(j_name+1, df.shape[1]):
            val = clean_num(df.iat[i_row, j])
            if pd.isna(val):
                continue
            col_label = header_string_for_col(df, j)
            # отсечём пустые/технические колонки
            if not col_label:
                continue
            unit = guess_unit_from_text(col_label) or unit_from_header
            recs.append({
                'kind': category_hint,
                'year': int(yy),
                'variable': col_label,
                'value': float(val),
                'unit': unit,
                'source_file': source_file,
                'sheet': sheet,
                'layout': 'single_year_from_filename'
            })
        reason = ''

    return recs, reason

def category_from_filename(name: str) -> str:
    n = name.lower()
    if 'объем' in n:
        return 'Объем промышленного производства (ВЭД)'
    if 'индекс промышленного производства по вэд' in n:
        return 'Индекс промышленного производства (ВЭД)'
    if 'индексы промышленного производства' in n:
        return 'Индексы ПП по видам деятельности'
    if 'снабжение электроэнерг' in n:
        return 'Снабжение: электроэнергия/газ/пар/ГВС/кондиц.'
    return 'Промышленность: прочее'

# --- сбор файлов
all_xls = sorted([p for p in raw.iterdir() if p.suffix.lower() in {'.xls', '.xlsx'}])
print('Найдено файлов:', len(all_xls))

rows = []
for p in all_xls:
    # ограничимся вашими 9 файлами (чтобы не цеплять чужие xls в папке)
    if not any(key in p.name for key in [
        'Объем промышленного производства', 'Индекс промышленного производства',
        'Индексы промышленного производства', 'Снабжение электроэнерг'
    ]):
        continue

    cat = category_from_filename(p.name)
    # движок
    engine = 'xlrd' if p.suffix.lower()=='.xls' else 'openpyxl'
    try:
        xls = pd.ExcelFile(p, engine=engine)
    except Exception:
        xls = pd.ExcelFile(p, engine=('openpyxl' if engine=='xlrd' else 'xlrd'))

    print(f'→ {p.name}: листов {len(xls.sheet_names)}')

    for sh in xls.sheet_names:
        # метаданные/легенды пропускаем
        if re.search(r'метадан|условн|legend|примеч', sh, flags=re.I):
            continue
        try:
            df = pd.read_excel(xls, sheet_name=sh, header=None, dtype=str)
        except Exception as e:
            print(f'   · пропуск «{sh}»: {e}')
            continue

        recs, reason = extract_sheet(df, p.name, sh, cat)
        if recs:
            print(f'   ✓ {sh}: {len(recs)} записей')
            rows += recs
        else:
            print(f'   · {sh}: {reason or "не распознали структуру"}')

if not rows:
    raise RuntimeError('Не удалось извлечь ни одной записи. '
                       'Если какой-то лист снова пропустился — кинь скрин верхней части таблицы (до строки «г. Алматы») + саму строку Алматы.')

df = (pd.DataFrame(rows)
        .drop_duplicates()
        .sort_values(['kind','year','source_file','sheet','variable'])
        .reset_index(drop=True))

save_path = out/'almaty_industry_all.csv'
df.to_csv(save_path, index=False, encoding='utf-8-sig')
print('✅ Сохранено:', save_path.resolve())
df.head(20)


Найдено файлов: 15
→ Индекс промышленного производства по ВЭД в разрезе регионов РК (январь-август 2025г.).xls: листов 3
   ✓ Лист3: 8 записей
→ Индексы промышленного производства по основным видам экономической деятельности в разрезе регионов (январь-декабрь 2024г.).xlsx: листов 12
   ✓ 01: 4 записей
   ✓ 02: 4 записей
   ✓ 03: 4 записей
   ✓ 04: 4 записей
   ✓ 05: 4 записей
   ✓ 06: 4 записей
   ✓ 07: 4 записей
   ✓ 08: 4 записей
   ✓ 09: 4 записей
   ✓ 10: 4 записей
   ✓ 11: 4 записей
   ✓ 12: 4 записей
→ Индексы промышленного производства по основным видам экономической деятельности в разрезе регионов.xlsx: листов 12
   · 01: нет годов в шапке и в названии файла/листа
   · 02: нет годов в шапке и в названии файла/листа
   · 03: нет годов в шапке и в названии файла/листа
   · 04: нет годов в шапке и в названии файла/листа
   · 05: нет годов в шапке и в названии файла/листа
   · 06: нет годов в шапке и в названии файла/листа
   · 07: нет годов в шапке и в названии файла/листа
   · 

,kind,year,variable,value,unit,source_file,sheet,layout
0,Индекс промышленного производства (ВЭД),2025,ИНДЕКС ПРОМЫШЛЕННОГО ПРОИЗВОДСТВА ПО ВИДАМ ЭК...,109.1,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
1,Индекс промышленного производства (ВЭД),2025,январь-август / 107.6 / 99.6 / 99.4 / 101.5 / ...,114.0,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
2,Индекс промышленного производства (ВЭД),2025,январь-апрель / 106.6 / 100.1 / 103 / 103.9 / ...,115.0,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
3,Индекс промышленного производства (ВЭД),2025,январь-июль / 106.9 / 99.8 / 101.2 / 101.1 / 1...,113.5,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
4,Индекс промышленного производства (ВЭД),2025,январь-июнь / 106.5 / 101.1 / 101.9 / 107.4 / ...,114.3,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
5,Индекс промышленного производства (ВЭД),2025,январь-май / 106.4 / 100.6 / 102 / 102.6 / 109...,114.1,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
6,Индекс промышленного производства (ВЭД),2025,январь-март / 106.7 / 100.3 / 104.6 / 102.8 / ...,111.2,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
7,Индекс промышленного производства (ВЭД),2025,январь-февраль / 105.9 / 99.1 / 105.3 / 102.3 ...,107.0,%,Индекс промышленного производства по ВЭД в раз...,Лист3,single_year_from_filename
8,Индексы ПП по видам деятельности,2024,Промышленность-всего / 103.4 / 106.7 / 111.3 /...,94.9,%,Индексы промышленного производства по основным...,01,single_year_from_filename
9,Индексы ПП по видам деятельности,2024,"водоснабжение; водоотведение; сбор, обработка ...",86.5,%,Индексы промышленного производства по основным...,01,single_year_from_filename
